<a href="https://colab.research.google.com/github/Ganesh-Mahato/Ganesh-Mahato/blob/main/PlantDiseaseDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mounting Dataset from google drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Installing tensorflow

In [7]:
!pip install transformers tensorflow


Importing necessary libraries

In [8]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from transformers import TFViTModel, ViTConfig


Setting dataset path

In [9]:
data_dir = "/content/drive/MyDrive/PlantVillage1"


Creating Training / Validation Generators

In [10]:
img_size = (224, 224)
batch_size = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

val_generator = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation",
    shuffle=True
)


Found 11380 images belonging to 9 classes.
Found 2840 images belonging to 9 classes.


Building the Hybrid CNN and ViT model

In [11]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from transformers import TFViTModel, ViTConfig

# Define a custom Keras Layer to wrap TFViTModel
class ViTLayer(layers.Layer):
    def __init__(self, vit_config, **kwargs):
        super().__init__(**kwargs)
        # Store config for serialization
        self.vit_config = vit_config
        # Instantiate the TFViTModel here
        self.vit_model = TFViTModel(vit_config)

    def call(self, inputs):
        # inputs shape: (batch, height, width, channels) -> (None, 224, 224, 3)
        # TFViTModel typically expects (batch, channels, height, width)
        # Permute to (batch, channels, height, width)
        permuted_inputs = tf.transpose(inputs, perm=[0, 3, 1, 2])
        # Pass the permuted KerasTensor inputs to the TFViTModel
        # The output is (last_hidden_state, pooled_output)
        # We need the pooled_output, which is at index 1
        return self.vit_model(permuted_inputs)[1]

    def get_config(self):
        config = super().get_config()
        config.update({
            "vit_config": self.vit_config.to_dict()
        })
        return config

    @classmethod
    def from_config(cls, config):
        # Reconstruct ViTConfig from its dictionary representation
        vit_config_dict = config.pop("vit_config")
        vit_config_obj = ViTConfig.from_dict(vit_config_dict)
        return cls(vit_config_obj, **config)

def build_hybrid_model(num_classes):

    # ---------------------------
    # Input layer
    # ---------------------------
    inputs = layers.Input(shape=(224, 224, 3))

    # ---------------------------
    # CNN Branch
    # ---------------------------
    cnn = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    cnn = layers.MaxPooling2D()(cnn)

    cnn = layers.Conv2D(64, 3, activation='relu', padding='same')(cnn)
    cnn = layers.MaxPooling2D()(cnn)

    cnn = layers.Conv2D(128, 3, activation='relu', padding='same')(cnn)
    cnn = layers.MaxPooling2D()(cnn)

    cnn = layers.Flatten()(cnn)
    cnn_features = layers.Dense(256, activation='relu')(cnn)

    # ---------------------------
    # Vision Transformer Branch
    # ---------------------------
    vit_config = ViTConfig(
        hidden_size=256,
        num_hidden_layers=6,
        num_attention_heads=8,
        intermediate_size=512
    )

    # Use the custom ViTLayer to integrate the TFViTModel
    vit_layer = ViTLayer(vit_config)
    vit_output = vit_layer(inputs)

    vit_features = layers.Dense(256, activation='relu')(vit_output)

    # ---------------------------
    # Feature Fusion
    # ---------------------------
    fused = layers.Concatenate()([cnn_features, vit_features])
    fused = layers.Dense(256, activation='relu')(fused)
    fused = layers.Dropout(0.3)(fused)

    outputs = layers.Dense(num_classes, activation='softmax')(fused)

    model = models.Model(inputs=inputs, outputs=outputs)
    return model

In [12]:
# def build_hybrid_model(num_classes):
#     from transformers import TFViTModel, ViTConfig
#     import tensorflow as tf
#     from tensorflow.keras import layers, Model

#     # --- FIX: Use a custom ViT model configured for Keras inputs ---
#     config = ViTConfig.from_pretrained(
#         "google/vit-base-patch16-224",
#         num_labels=num_classes,
#         hidden_dropout_prob=0.1,
#         attention_probs_dropout_prob=0.1,
#     )

#     vit = TFViTModel(config)

#     # Keras input
#     inputs = layers.Input(shape=(224, 224, 3))

#     # Preprocessing for ViT
#     x = layers.Rescaling(1/255)(inputs)
#     x = layers.Lambda(lambda img: tf.image.resize(img, (224, 224)))(x)

#     # ViT expects (batch, 3, 224, 224) for channels-first
#     x = layers.Permute((3, 1, 2))(x)

#     vit_outputs = vit(pixel_values=x).last_hidden_state
#     vit_features = layers.GlobalAveragePooling1D()(vit_outputs)

#     # CNN branch
#     cnn = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
#     cnn = layers.MaxPooling2D()(cnn)
#     cnn = layers.Conv2D(64, (3, 3), activation='relu')(cnn)
#     cnn = layers.GlobalAveragePooling2D()(cnn)

#     # Combine
#     combined = layers.concatenate([vit_features, cnn])
#     combined = layers.Dense(256, activation='relu')(combined)
#     outputs = layers.Dense(num_classes, activation='softmax')(combined)

#     return Model(inputs, outputs)


Building and Compiling Model

In [13]:
num_classes = train_generator.num_classes

model = build_hybrid_model(num_classes)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 224, 224,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 112, 112,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 56, 56,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 56, 56,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 28, 28,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 100352)    │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vi_t_layer          │ (None, 256)       │          0 │ input_layer[0][0] │
│ (ViTLayer)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │ 25,690,368 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │     65,792 │ vi_t_layer[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 512)       │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 256)       │    131,328 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 9)         │      2,313 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 25,983,049 (99.12 MB)

 Trainable params: 25,983,049 (99.12 MB)

 Non-trainable params: 0 (0.00 B)

Training Model

In [14]:
history = model.fit(
    train_generator,
    epochs=2,
    validation_data=val_generator
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/2
356/356 ━━━━━━━━━━━━━━━━━━━━ 4127s 12s/step - accuracy: 0.4482 - loss: 1.5686 - val_accuracy: 0.7996 - val_loss: 0.6197
Epoch 2/2
356/356 ━━━━━━━━━━━━━━━━━━━━ 2535s 7s/step - accuracy: 0.7902 - loss: 0.6175 - val_accuracy: 0.8437 - val_loss: 0.4495


In [15]:
import gc
gc.collect()
tf.keras.backend.clear_session()

Loading class label

In [22]:
import json

with open("/content/drive/MyDrive/class_labels.json", "w") as f:
    json.dump(train_generator.class_indices, f)


Saving Model

In [23]:
import tensorflow as tf

# model.save("/content/drive/MyDrive/plant_disease_cnn_vit.h5")
# model.save("/content/drive/MyDrive/plant_disease_cnn_vit.keras")
model.save("/content/drive/MyDrive/plant_disease_hybrid.keras")
print("MODEL SAVED SUCCESSFULLY!")

MODEL SAVED SUCCESSFULLY!


Loading Model and labels

In [25]:
# ---------------------------------------------------
# LOAD SAVEDMODEL + CLASS LABELS
# ---------------------------------------------------
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
import json

# Load saved model
saved_model_path = "/content/drive/MyDrive/plant_disease_hybrid.keras"
loaded_model = tf.keras.models.load_model(saved_model_path, custom_objects={'ViTLayer': ViTLayer})

# Load class names
with open("/content/drive/MyDrive/class_labels.json", "r") as f:
    class_indices = json.load(f)

idx_to_class = {v: k for k, v in class_indices.items()}

Prediction Helper Function

In [29]:
# Preprocess image
def preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_arr = image.img_to_array(img) / 255.0
    img_arr = np.expand_dims(img_arr, axis=0)
    return img_arr

# Predict function
def predict_plant_disease(img_path):
    img = preprocess_image(img_path)

    img_tensor = tf.convert_to_tensor(img, dtype=tf.float32)

    # Run inference using the loaded_model
    pred = loaded_model(img_tensor)

    # Extract output tensor
    output_tensor = pred.numpy()

    class_id = np.argmax(output_tensor)
    confidence = np.max(output_tensor)

    class_name = idx_to_class[class_id]

    return class_name, confidence

Text Prediction Block

In [30]:
# Test image path
test_img = "/content/drive/MyDrive/PlantVillage/Pepper__bell___healthy/9439a69a-66c9-48cc-8030-7ac35a92e2a2___JR_HL 8577.JPG"  # Change this

predicted_class, confidence = predict_plant_disease(test_img)

print("Predicted Class:", predicted_class)
print("Confidence:", confidence)

Predicted Class: Tomato_Septoria_leaf_spot
Confidence: 0.906943
